# **Initialization**

In [ ]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from docplex.mp.model import Model

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis_implementation\3_Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis_implementation\3_Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Data**

In [18]:
# ==========================================
# 1. DATA READING
# ==========================================
def read_tsp_cappart_format(file_path):
    """Parses TSP text files."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
        
    with open(file_path, 'r') as f:
        values = f.read().split()

    iterator = iter(values)
    try:
        n = int(next(iterator))
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator))
                row.append(int(val))
            c.append(row)
        return n, c
    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

# Load Data Global Variables (simplest way for registries to access them)
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20\0.txt"

try:   
    number_of_customers, distance_list = read_tsp_cappart_format(base_path)
    print(f'Number of customer is {number_of_customers}')
    print(f'Distance matrix is {distance_list}')

except FileNotFoundError:
    print(f"Error: The file at {base_path} was not found.")

Number of customer is 20
Distance matrix is [[0, 34, 48, 30, 50, 10, 28, 41, 32, 4, 53, 62, 22, 59, 38, 36, 36, 37, 2, 13], [34, 0, 33, 4, 83, 34, 42, 69, 56, 35, 63, 30, 44, 39, 49, 5, 53, 70, 35, 31], [48, 33, 0, 35, 94, 41, 70, 90, 49, 46, 40, 55, 67, 72, 33, 28, 80, 73, 50, 54], [30, 4, 35, 0, 79, 30, 38, 65, 53, 31, 62, 34, 40, 40, 48, 8, 50, 66, 31, 27], [50, 83, 94, 79, 0, 54, 54, 28, 53, 50, 81, 109, 45, 97, 70, 86, 50, 27, 48, 54], [10, 34, 41, 30, 54, 0, 38, 50, 24, 7, 44, 64, 32, 64, 28, 35, 46, 37, 12, 22], [28, 42, 70, 38, 54, 38, 0, 31, 60, 32, 82, 58, 10, 43, 66, 47, 11, 56, 27, 16], [41, 69, 90, 65, 28, 50, 31, 0, 61, 44, 88, 90, 26, 73, 74, 73, 24, 43, 40, 38], [32, 56, 49, 53, 53, 24, 60, 61, 0, 28, 28, 86, 51, 89, 18, 55, 66, 27, 33, 45], [4, 35, 46, 31, 50, 7, 32, 44, 28, 0, 50, 64, 25, 62, 34, 36, 40, 36, 5, 17], [53, 63, 40, 62, 81, 44, 82, 88, 28, 50, 0, 91, 75, 101, 16, 60, 90, 54, 55, 66], [62, 30, 55, 34, 109, 64, 58, 90, 86, 64, 91, 0, 65, 27, 79, 32, 68, 99,

# **Model and dual bound declaration**

In [19]:
# ==========================================
# 2. DEFINE DIDP MODEL (The "Physics" of TSP)
# ==========================================
def creation_of_didp_model_function():
    n = number_of_customers
    c = distance_list
    
    model = m_dp.Model(maximize=False, float_cost=False)
    customer = model.add_object_type(number=n)

    # State Variables
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    location = model.add_element_var(object_type=customer, target=0)
    travel_time = model.add_int_table(c)

    # Transitions
    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[(unvisited, unvisited.remove(j)), (location, j)],
        )
        model.add_transition(visit)

    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
        effects=[(location, 0)],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)
    model.add_base_case([unvisited.is_empty(), location == 0])

    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
    }
    
    return (model, metadata)

# ==========================================
# 3. DEFINE DUAL BOUNDS (The "Heuristics")
# ==========================================
def dual_bound_expression_function(didp_bundle):
    model, metadata = didp_bundle
    dist_matrix = np.array(metadata['distance_matrix'])
    unvisited_var = metadata['unvisited_var']
    
    # --- Define your Heuristics here ---
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = dist_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = dist_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    # Automatically grab all functions starting with "h_" in this scope
    return automatic_creation_of_dual_bounds_registry(locals())

print("✅ TSP Problem definitions loaded!")

✅ TSP Problem definitions loaded!


# **Execution**

In [ ]:
# ==========================================
# 4. CONFIGURE AND RUN
# ==========================================

# 1. Define Parameters
# Note: We map 'optimal_cost_reference' -> 'reference_point' to match the library definition
params = EAHyperparameters(
    # --- 1. Population ---
    population_size=20,          
    generations=5,
    crossover_rate=0.8,
    mutation_rate=0.2,
    elitism_rate=0.02,           # Added missing parameter

    # --- 2. Ranges & Constraints ---
    lb_range_of_constant=0.5,
    ub_range_of_constant=5.0,
    min_chromosome_length=2,     # Added missing parameter
    max_chromosome_length=10,    # Added missing parameter

    # --- 3. Operator Specifics ---
    tournament_size=5,                             # Added missing parameter
    tournament_probability=0.8,                    # Added missing parameter
    mutation_max_subtree_depth=5,                  # Added missing parameter
    homology_1_point_crossover_probability=0.5,    # Added missing parameter
    subtree_crossover_probability=0.9,             # Added missing parameter
    uniform_crossover_probability=0.5,             # Added missing parameter

    # --- 4. Problem Specific ---
    reference_point=400,         # <--- CHANGED FROM 'optimal_cost_reference'
    solver_time_limit=2.0,
    
    # Optional: You can override available operations if needed
    available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
)

print("--- Starting TSP Test Run ---")

# 2. Execution
best_individual = evolution_algorithm_execution(
    didp_model_registry=creation_of_didp_model_function,
    dual_bound_expression_function=dual_bound_expression_function,
    params=params
)

--- Starting TSP Test Run ---
--- Initialization: Generating Population of size 20 - 5 generations ---
Generating Initial Population at time: Sat Dec 13 00:12:14 2025
Initial Population Generated at time: Sat Dec 13 00:12:55 2025
Initial Best Fitness: 0.0
Gen 1: Best Fitness = 0.0 | Global Best = 0.0
Gen 2: Best Fitness = 0.0 | Global Best = 0.0
Gen 3: Best Fitness = 0.0 | Global Best = 0.0
Gen 4: Best Fitness = 0.0 | Global Best = 0.0
Gen 5: Best Fitness = 0.0 | Global Best = 0.0

       PERFORMANCE PROFILING REPORT       
Total Runtime:    235.7145 seconds
--- Evolution Completed ---



In [20]:
# 1. Create the Missing Registry
# We must instantiate the model once to get the heuristic functions dictionary
temp_bundle = creation_of_didp_model_function()
dual_bound_functions_registry = dual_bound_expression_function(temp_bundle)

# 2. Setup Chromosome
combined_dual_bound_chromosome = [5.14, 'h_mst', 'MULTIPLY']
temp_dict = {'chromosome': combined_dual_bound_chromosome, 'fitness': 0}

print("The combined dual bounds will be depicted in the following code")
combined_dual_bound_function = compile_chromosome_to_useable_function(
    temp_dict, 
    dual_bound_functions_dict=dual_bound_functions_registry,
    print_code=True
)

print("\nSuccessfully created solver. Starting search...\n")

# 3. Run Solver
result = combining_modified_didppy_solver_with_chromosome(
    combined_dual_bound_chromosome, 
    creation_of_didp_model_function, 
    dual_bound_expression_function, 
    solver_time_limit=10,
    output_other_result=True,
    print_timing_stats=True
)

# If in Jupyter, use display(result), otherwise print(result)
try:
    display(result)
except NameError:
    print(result)

The combined dual bounds will be depicted in the following code
Generated Code:
def dual_bound_combination(state):
    return (5.14 * h_mst(state))


Successfully created solver. Starting search...


                    📋 SOLVER RUN REPORT                     
🎯 SOLUTION STATUS:
   • Cost:              431
   • Status:            ⚠️  Suboptimal / Timeout
   • Nodes Generated:   9,259
   • Nodes Expanded:    2,919
   • Branching Factor:  ~3.17
------------------------------------------------------------
⏱️  TIME DISTRIBUTION (Total: 10.0350s):
   [Pure CABS time:   9.7%] 🆚 [Bridge time:  90.3%]

   1. 🟢 Pure CABS (Search):    0.9767 s
      └─ Average time per expanded state:   0.3346 ms
   2. 🔴 Total Bridge Time:     9.0583 s
      ├─ 🐍 Dual Bound Calculation Time: :     8.7060 s  ( 96.1% of bridge time)
      └─ 🌉 Switching Time:  0.3523 s  (  3.9% of bridge time)
------------------------------------------------------------
📊 PYTHON DUAL BOUND CALL STATS:
   • Total Calls:       31,05

None